# Automação de Extração de Relatório do Contas a Pagar no NetSuite

- Extrair do sistema NetSuite o relatório de pagamento pelo Contas a Pagar
- Tratamento da base CONTAS A PAGAR, padronização de nomes de colunas e formatos de dados, além da atualização da planilha CONTAS A PAGAR.xlsx e Power BI - CONTAS A PAGAR

# Importando as Bibliotecas

In [5]:
import gc # Importa o garbage collector
import logging
import numpy as np
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import sys
import time
import tkinter as tk
import win32com.client
from asyncio.log import logger
from contextlib import contextmanager
import datetime
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [6]:
CONFIG = {
    'id_processo': 12,
    'processos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\NETSUITE - CP.xlsx'),
    'fornecedores': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\FORNECEDORES.xlsx'),
    'padrao': 'RegistrodeC_P',
}

COLUNAS_SAIDA = [
    'Conta',
    'Tipo',
    'Data',
    'Numero_documento',
    'Id_fornecedor',
    'Memorando',
    'Faturado',
    'Pago'
]

# Registra etapa do processo

In [7]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais

In [8]:
automacao = r'X:\Gestao_de_Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\AUTOMACAO'

chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automacao}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get("https://5650531.app.netsuite.com/app/login/secure/enterpriselogin.nl?c=5650531&whence=")
    wait = WebDriverWait(driver, 150)
    actions = ActionChains(driver)
    
    print("⏳ Sincronizando NetSuite...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

2026-01-23 10:02:38,813 - INFO - ====== WebDriver manager ======
2026-01-23 10:02:41,491 - INFO - Get LATEST chromedriver version for google-chrome
2026-01-23 10:02:41,958 - INFO - Get LATEST chromedriver version for google-chrome
2026-01-23 10:02:42,431 - INFO - Driver [C:\Users\rodrigo.bernandes\.wdm\drivers\chromedriver\win64\143.0.7499.192\chromedriver-win32/chromedriver.exe] found in cache


⏳ Sincronizando NetSuite...
🏁 Acesso finalizado


# Acessando o NetSuite

In [9]:
pyautogui.PAUSE = 1

# Área de login no NetSuite
pyautogui.click(x=543, y=392)
pyautogui.write("rodrigo.bernardes@afpesp.org.br")
pyautogui.press("tab")
pyautogui.write("Spfc1935!))")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(5) # Inserir o token

root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Acesso ao NetSuite realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

 ✅ Continuando o processo...
----------------------------------------------------------------------------------------------------

   ✅ Acesso ao NetSuite realizado com sucesso

----------------------------------------------------------------------------------------------------


# Dentro do NetSuite, acessando o Relatório do Contas a Pagar

In [ ]:
# Acessando o C/P
time.sleep(5) # Tempo para aguardar carregar a página
pyautogui.click(x=711, y=222) # Relatórios
time.sleep(2)
pyautogui.click(x=773, y=656) # Fornecedores/contas a pagar
time.sleep(1)
pyautogui.click(x=997, y=658) # Registro de C/P
time.sleep(7)
pyautogui.click(x=440, y=795) # Inserir a data de hoje
time.sleep(1)

# Data atual
data_atual = datetime.date.today()

# Formato brasileiro: DD/MM/AAAA
fmt_br = "%d/%m/%Y"
data_atual_br = data_atual.strftime(fmt_br)

pyautogui.write(data_atual_br) # Digitar Data início
time.sleep(1)
pyautogui.click(x=45, y=830) # Atualizar
time.sleep(12)
pyautogui.click(x=1460, y=834) # Exportar - CSV
time.sleep(5)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Relatório Contas a Pagar extraído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Relatório Contas a Pagar extraído

----------------------------------------------------------------------------------------------------


# Configuração de Logging

In [11]:
# REMOVER qualquer arquivo de log existente
log_file = Path('netsuite_cp.log')
if log_file.exists():
    try:
        log_file.unlink()
    except Exception as e:
        print(f"Erro ao remover arquivo de log existente: {e}")

# RESETAR logging (remove todos os handlers antigos)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configurar logging APENAS no console (SEM arquivo)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False

# Remover handlers antigos do logger
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Adicionar APENAS StreamHandler (console)
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)


# Carga e tratamento da base

In [13]:
from datetime import datetime

# --- Funções Auxiliares ---
@contextmanager
def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operacoes seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)
        wb.close()

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    try:
        with gerenciar_workbook(caminho, 'REGISTROS') as ws:
            ws.append([id_proc, datetime.today(), etapa])
        logger.info(f"[OK] Etapa {etapa} registrada com sucesso.")
    except Exception as e:
        logger.error(f"[ERRO] Falha ao registrar etapa {etapa}: {e}")

def carregar_de_para_fornecedores(caminho: Path) -> pd.DataFrame:
    """Carrega e processa de-para de fornecedores."""
    try:
        df = pd.read_excel(caminho)
        
        if 'ID' not in df.columns or 'NOME CADASTRO' not in df.columns:
            logger.warning("[AVISO] Colunas 'ID' ou 'NOME CADASTRO' nao encontradas no arquivo de fornecedores.")
            return pd.DataFrame() # Retorna DataFrame vazio se colunas essenciais faltarem
        
        df = df[['ID', 'NOME CADASTRO']].copy()
        df.columns = ['ID', 'Nome_cadastro']
        
        return df
    except FileNotFoundError:
        logger.error(f"[ERRO] Arquivo de fornecedores nao encontrado: {caminho}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"[ERRO] Erro ao carregar de-para de fornecedores: {e}")
        return pd.DataFrame()

def converter_moeda(valor_str: str) -> float:
    """Converte string de moeda R$X.XXX,XX para float."""
    if pd.isna(valor_str) or valor_str == '':
        return 0.0
    
    try:
        valor = str(valor_str).replace('R$', '').replace('.', '').replace(',', '.')
        return float(valor)
    except ValueError:
        return 0.0
    except Exception as e:
        logger.warning(f"[AVISO] Erro ao converter valor '{valor_str}' para moeda: {e}. Retornando 0.0.")
        return 0.0

def processar_arquivo(caminho: Path, de_para: pd.DataFrame) -> pd.DataFrame:
    """Processa um arquivo NetSuite CP."""
    try:
        # --- 1. Leitura do CSV com fallback de encoding ---
        try:
            df = pd.read_csv(caminho, skiprows=6, on_bad_lines='skip', encoding='utf-8')
            logger.info(f"   [OK] Arquivo lido com encoding 'utf-8'.")
        except UnicodeDecodeError:
            df = pd.read_csv(caminho, skiprows=6, on_bad_lines='skip', encoding='latin-1')
            logger.info(f"   [OK] Arquivo lido com encoding 'latin-1' (fallback).")
        
        if df.empty:
            logger.warning(f"   [AVISO] Arquivo '{caminho.name}' esta vazio apos leitura.")
            return None

        # --- 2. Limpeza e Renomeação de Colunas ---
        df.columns = df.columns.str.strip() # Remove espaços em branco de todos os nomes de coluna
        
        mapeamento_colunas = {}
        for col in df.columns:
            col_lower = col.lower()
            # Renomear "Número do documento" para "Numero_documento"
            if 'nmero' in col_lower or 'número' in col_lower or 'documento' in col_lower:
                mapeamento_colunas[col] = 'Numero_documento'
            # Outras colunas que podem ter problemas de espaço ou encoding
            elif col_lower == 'data':
                mapeamento_colunas[col] = 'Data'
            elif col_lower == 'faturado':
                mapeamento_colunas[col] = 'Faturado'
            elif col_lower == 'pago':
                mapeamento_colunas[col] = 'Pago'
            elif col_lower == 'fornecedor':
                mapeamento_colunas[col] = 'Fornecedor'
            elif col_lower == 'memorando':
                mapeamento_colunas[col] = 'Memorando'
            elif col_lower == 'conta':
                mapeamento_colunas[col] = 'Conta'
            elif col_lower == 'tipo':
                mapeamento_colunas[col] = 'Tipo'

        if mapeamento_colunas:
            df = df.rename(columns=mapeamento_colunas)
            logger.info(f"   [OK] Colunas renomeadas: {mapeamento_colunas}")
        
        # Remover colunas duplicadas que podem ter surgido
        df = df.loc[:, ~df.columns.duplicated(keep='first')]
        logger.info(f"   [OK] Colunas após limpeza de duplicatas: {df.columns.tolist()}")

        # --- 3. Filtragem da Coluna 'Conta' ---
        if 'Conta' in df.columns:
            linhas_antes = len(df)
            # Remove apenas linhas onde 'Conta' começa com 'Total' (case-insensitive, strip spaces)
            df = df[~df['Conta'].astype(str).str.strip().str.lower().str.startswith('total', na=False)].copy()
            logger.info(f"   [OK] Removidas {linhas_antes - len(df)} linhas de totais da coluna 'Conta'.")
        else:
            logger.warning(f"   [AVISO] Coluna 'Conta' nao encontrada para filtragem no arquivo '{caminho.name}'.")

        # --- 4. Conversão de Tipos de Dados ---
        if 'Data' in df.columns:
            df['Data'] = pd.to_datetime(df['Data'], format='%d/%m/%Y', errors='coerce')
        if 'Faturado' in df.columns:
            df['Faturado'] = df['Faturado'].apply(converter_moeda)
        if 'Pago' in df.columns:
            df['Pago'] = df['Pago'].apply(converter_moeda)

        # --- 5. Merge com Fornecedores ---
        if 'Fornecedor' in df.columns and not de_para.empty:
            df = df.merge(de_para, left_on='Fornecedor', right_on='Nome_cadastro', how='left', suffixes=('', '_de_para'))
            
            # Limpar colunas com sufixo '_de_para' que podem ter sido criadas
            for col in df.columns:
                if col.endswith('_de_para'):
                    df = df.drop(columns=[col])
            
            df = df.rename(columns={'ID': 'Id_fornecedor'})
            df = df.drop('Nome_cadastro', axis=1, errors='ignore')
            logger.info(f"   [OK] Merge com fornecedores realizado. Colunas: {df.columns.tolist()}")
        elif 'Fornecedor' not in df.columns:
            logger.warning(f"   [AVISO] Coluna 'Fornecedor' nao encontrada no arquivo '{caminho.name}' para merge.")
        elif de_para.empty:
            logger.warning(f"   [AVISO] DataFrame de fornecedores vazio, merge nao realizado.")

        # --- 6. Limpeza Final e Preenchimento de Nulos ---
        df = df.loc[:, ~df.columns.duplicated(keep='first')] # Última limpeza de duplicatas
        df = df.fillna(0) # Preenche nulos com 0 (para numéricos)
        df = df.replace({pd.NA: None, np.nan: None, '<NA>': None, 'nan': None}) # Limpeza de outros tipos de nulos

        # --- 7. Seleção e Ordenação das Colunas Finais ---
        final_df_columns = []
        for col in COLUNAS_SAIDA:
            if col in df.columns:
                final_df_columns.append(col)
            else:
                # Adiciona colunas faltantes com valores padrão
                if col in ['Id_fornecedor', 'Faturado', 'Pago']:
                    df[col] = 0
                else:
                    df[col] = None
                final_df_columns.append(col)
                logger.warning(f"   [AVISO] Coluna '{col}' nao encontrada e adicionada com valores padrao.")
        
        df = df[final_df_columns] # Garante a seleção e ordem das colunas
        
        logger.info(f"   [OK] Processamento concluido para '{caminho.name}' com {len(df)} registros.")
        gc.collect() # Libera memória após processar cada arquivo
        return df
        
    except Exception as e:
        logger.error(f"   [ERRO] ERRO geral ao processar '{caminho.name}': {e}", exc_info=True)
        return None

def validar_novos_fornecedores(df: pd.DataFrame, de_para: pd.DataFrame) -> dict:
    """Valida se ha novos fornecedores nao cadastrados."""
    logger.info("\n[ETAPA] Validando fornecedores...")
    
    if 'Id_fornecedor' not in df.columns:
        logger.warning("[AVISO] Coluna 'Id_fornecedor' nao encontrada para validacao de fornecedores.")
        return {
            'total_registros': len(df),
            'registros_com_fornecedor': 0,
            'registros_sem_fornecedor': len(df),
            'novos_fornecedores_unicos': 0,
            'passou_validacao': False
        }

    novos_fornecedores = df[df['Id_fornecedor'] == 0.0].copy()
    
    validacao = {
        'total_registros': len(df),
        'registros_com_fornecedor': len(df[df['Id_fornecedor'] != 0.0]),
        'registros_sem_fornecedor': len(novos_fornecedores),
        'novos_fornecedores_unicos': novos_fornecedores['Fornecedor'].nunique() if 'Fornecedor' in novos_fornecedores.columns else 0,
        'passou_validacao': len(novos_fornecedores) == 0
    }
    
    logger.info(f"   [OK] Total de registros: {validacao['total_registros']}")
    logger.info(f"   [OK] Registros com fornecedor identificado: {validacao['registros_com_fornecedor']}")
    logger.info(f"   [OK] Registros sem fornecedor: {validacao['registros_sem_fornecedor']}")
    
    if validacao['novos_fornecedores_unicos'] > 0:
        logger.warning(f"\n   [AVISO] ATENCAO: {validacao['novos_fornecedores_unicos']} novo(s) fornecedor(es) encontrado(s)!")
        logger.warning("   [AVISO] Fornecedores nao cadastrados:")
        
        fornecedores_nao_cadastrados = novos_fornecedores['Fornecedor'].unique()
        for forn in fornecedores_nao_cadastrados:
            count = len(novos_fornecedores[novos_fornecedores['Fornecedor'] == forn])
            logger.warning(f"      * {forn} ({count} registro(s))")
    else:
        logger.info("\n   [OK] Validacao concluida: NENHUM novo fornecedor para cadastrar.")
        logger.info("   [OK] Todos os fornecedores estao devidamente cadastrados.")
    
    return validacao

def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"\n[ETAPA] Criando Excel final ({len(df)} registros)...")
    
    try:
        df.to_excel(caminho, sheet_name='NETSUITE - CP', index=False, engine='openpyxl')
        
        wb = load_workbook(caminho)
        ws = wb.active
        
        tabela = Table(displayName="NETSUITE_CP", ref=ws.dimensions)
        estilo = TableStyleInfo(
            name="TableStyleLight13",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=True
        )
        tabela.tableStyleInfo = estilo
        ws.add_table(tabela)
        
        wb.save(caminho)
        wb.close()
        logger.info(f"[OK] Excel criado com sucesso em: {caminho}")
    except Exception as e:
        logger.error(f"[ERRO] Falha ao criar arquivo Excel: {e}", exc_info=True)
    finally:
        gc.collect() # Libera memória após criar o Excel

# --- Main Execution Block ---
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("[INICIO] PROCESSAMENTO NETSUITE - CP")
    logger.info("=" * 80)
    
    try:
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
        logger.info("\n[ETAPA 1] Buscando arquivos...")
        arquivos = sorted([f for f in CONFIG['origem'].iterdir() 
                          if f.name.startswith(CONFIG['padrao']) and f.suffix.lower() == '.csv'])
        
        if not arquivos:
            logger.error("[ERRO] Nenhum arquivo CSV encontrado na pasta de origem!")
            sys.exit(1) # Sai do script com erro
        
        logger.info(f"[OK] Encontrados {len(arquivos)} arquivo(s) CSV.")
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
        logger.info("\n[ETAPA 2] Carregando de-para de fornecedores...")
        de_para = carregar_de_para_fornecedores(CONFIG['fornecedores'])
        if de_para.empty:
            logger.warning("[AVISO] Nao foi possivel carregar a base de fornecedores ou ela esta vazia.")
        else:
            logger.info(f"[OK] Carregados {len(de_para)} fornecedores.")
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
        logger.info("\n[ETAPA 3] Processando arquivos...")
        
        todas_bases = []
        
        for idx, arquivo in enumerate(arquivos, 1):
            logger.info(f"\n   [ETAPA] Processando [{idx}/{len(arquivos)}] '{arquivo.name}'...")
            
            df = processar_arquivo(arquivo, de_para)
            
            if df is not None and not df.empty:
                todas_bases.append(df)
                logger.info(f"   [OK] {len(df)} registros adicionados de '{arquivo.name}'.")
                
                try:
                    destino = CONFIG['movidos'] / arquivo.name
                    shutil.move(str(arquivo), str(destino))
                    logger.info(f"   [OK] Arquivo '{arquivo.name}' movido para '{CONFIG['movidos'].name}'.")
                except Exception as e:
                    logger.warning(f"   [AVISO] Erro ao mover arquivo '{arquivo.name}': {e}")
            else:
                logger.warning(f"   [AVISO] Nenhum dado valido processado para '{arquivo.name}'.")
            gc.collect() # Limpa memória após cada iteração do loop
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 3)
        
        logger.info("\n[ETAPA 4] Consolidando dados...")
        
        if todas_bases:
            base_final = pd.concat(todas_bases, ignore_index=True)
            base_final = base_final.drop_duplicates()
            logger.info(f"[OK] {len(base_final)} registros consolidados e duplicatas removidas.")
            
            criar_excel_com_tabela(base_final, CONFIG['saida'])
            
            validacao = validar_novos_fornecedores(base_final, de_para)
            
            tempo_duracao = datetime.now() - tempo_inicio
            
            logger.info("\n" + "=" * 80)
            logger.info("[SUCESSO] PROCESSO FINALIZADO COM SUCESSO!")
            logger.info("=" * 80)
            logger.info(f"\n[RESUMO EXECUTIVO]:")
            logger.info(f"   * Arquivos processados: {len(arquivos)}")
            logger.info(f"   * Total de registros na base final: {validacao['total_registros']}")
            logger.info(f"   * Registros com fornecedor identificado: {validacao['registros_com_fornecedor']}")
            logger.info(f"   * Registros sem fornecedor: {validacao['registros_sem_fornecedor']}")
            
            if validacao['passou_validacao']:
                logger.info(f"\n[OK] VALIDACAO CONCLUIDA: Nenhum novo fornecedor para cadastrar.")
                logger.info(f"   Todos os arquivos foram movidos e tratados com sucesso.")
            else:
                logger.warning(f"\n[AVISO] ATENCAO: {validacao['novos_fornecedores_unicos']} novo(s) fornecedor(es) requer(em) cadastro.")
            
            logger.info(f"\n   Tempo de execucao: {tempo_duracao}")
            logger.info("=" * 80)
            
        else:
            logger.error("[ERRO] Nenhum arquivo foi processado ou resultou em dados validos para consolidacao!")
            sys.exit(1) # Sai do script com erro
        
    except Exception as e:
        logger.error(f"\n[ERRO CRITICO] Ocorreu um erro inesperado: {e}", exc_info=True)
        sys.exit(1) # Sai do script com erro
    finally:
        gc.collect() # Limpa memória no final da execução

2026-01-23 10:04:39,498 - INFO - ================================================================================
2026-01-23 10:04:39,499 - INFO - [INICIO] PROCESSAMENTO NETSUITE - CP
2026-01-23 10:04:39,500 - INFO - ================================================================================
2026-01-23 10:04:39,610 - INFO - [OK] Etapa 0 registrada com sucesso.
2026-01-23 10:04:39,610 - INFO - 
[ETAPA 1] Buscando arquivos...
2026-01-23 10:04:39,610 - INFO - [OK] Encontrados 1 arquivo(s) CSV.
2026-01-23 10:04:39,729 - INFO - [OK] Etapa 1 registrada com sucesso.
2026-01-23 10:04:39,730 - INFO - 
[ETAPA 2] Carregando de-para de fornecedores...
2026-01-23 10:04:39,810 - INFO - [OK] Carregados 177 fornecedores.
2026-01-23 10:04:39,897 - INFO - [OK] Etapa 2 registrada com sucesso.
2026-01-23 10:04:39,897 - INFO - 
[ETAPA 3] Processando arquivos...
2026-01-23 10:04:39,898 - INFO - 
   [ETAPA] Processando [1/1] 'RegistrodeC_P-174.csv'...
2026-01-23 10:04:39,910 - INFO -    [OK] Arquivo lid

# Atualizando o arquivo Excel CONTAS A PAGAR
- X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS

In [14]:
# Caminho do arquivo
path_excel = r"X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CONTAS A PAGAR.xlsx"
os.startfile(path_excel) # Abre o arquivo pelo Windows
time.sleep(5)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)
pyautogui.write('Contas_a_Pagar') # Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)
pyautogui.hotkey('alt', 'F5') #Atualizando o arquivo
time.sleep(7)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
pyautogui.write('Tab.Din') # Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)
pyautogui.hotkey('alt', 'F5') #Atualizando o arquivo
time.sleep(3)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
pyautogui.write('Media_Movel') # Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

# Aguardando informar o HC atual para poder calcular o Custo Médio
root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

pyautogui.hotkey('ctrl', 'b') # Salvar o arquivo
time.sleep(4)
pyautogui.hotkey('Alt', 'F4') # Fechar o arquivo
time.sleep(3)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

 ✅ Continuando o processo...
----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Atualização do Power BI - CONTAS A PAGAR

In [15]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\CONTAS A PAGAR.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(40)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(40)
pyautogui.click(x=1294, y=109) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.click(x=712, y=379) # Clicar em GESTÃO DE PESSOAS
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(5)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI Atualizado

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [16]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:05:35.744824

----------------------------------------------------------------------------------------------------
